In [1]:
from typing import Annotated

from langchain_groq import ChatGroq
from langchain_core.messages import AnyMessage, AIMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage

c:\Users\ChaudhariMuskan\Aditi_Projects\7-Weeks-Daily-Progress\Week-6\LangGraph\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model="llama-3.3-70b-versatile")

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):

    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })

    if decision["approved"] == 'no':
        return {"messages": [AIMessage(content="Not approved")]}
    else:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

In [ ]:
# Graph -> START->chat->END
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

checkpointer = MemorySaver()

app = builder.compile(checkpointer=checkpointer)

In [ ]:
app

In [ ]:
config = {"configurable": {"thread_id": '1234'}}

initial_input = {
    "messages":[
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

result = app.invoke(initial_input, config=config)

In [ ]:
result

In [ ]:
message = result['__interrupt__'][0].value
message

In [ ]:
user_input = input(f"\nBackend message - {message} \n Approve this question? (y/n): ")

In [ ]:
final_result = app.invoke(
    Command(resume={"approved": user_input}),
    config=config,
)

In [ ]:
print(final_result["messages"[-1]].content)